# Stream on Colab (T4 GPU)

Byte-level SSM language model — token-free, position-free, O(n) memory —
A/B-tested against a byte-level GPT baseline at matched scale.

**Models:** Stream-10M (pure SSM) · StreamR-10M (+ sparse-retrieval blocks, v2) ·
StreamD-10M (+ gated delta memory) · GPT-8L (reference competitor).

**What this notebook does**

1. **Cell 1** checks the runtime (needs a T4 GPU) and force-updates a fresh
   clone of `origin/main`.
2. **Cell 2** is fully self-contained: it embeds and rewrites `model.py` +
   `delta_scan.py` (no dependency on the clone being current), verifies
   correctness (`check_retrieval_v2`, `check_delta`) *before* any training,
   verifies and enables both fused Triton scans (SSM scan + gated-delta scan;
   automatic fallback to the eager loops if anything fails), downloads and
   prepares TinyStories bytes if missing, then trains every model on identical
   batches and prints val losses, per-step medians, and automatic verdicts.

**Legs:** leg A (quality) `T=4096, B=4, 1200 steps`; leg B (long context)
`T=16384, B=2, 400 steps`. GPT-8L runs only in leg A — its O(n²) attention
does not fit a T4 at 16K, which is precisely the gap Stream closes.

**Runtime:** ~45–60 min on a free T4. Progress logs every ~100 steps.

> **Something failed?** Paste the full traceback plus the `[checks]` lines
> from cell 2 into the chat with the error.


In [ ]:
# @title 1. Environment check + clone repo (fresh from origin/main)
import sys, os

REPO_URL = 'https://github.com/nishantXnova/RETRANS-X.git'
PROJECT_DIR = '/content/RETRANS-X'

import torch
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')
if torch.cuda.device_count() == 0:
    raise SystemExit('No GPU detected. Use Runtime > Change runtime type > T4 GPU.')
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name} | VRAM {p.total_memory/1e9:.1f} GB')

if not os.path.isdir(PROJECT_DIR):
    print('Cloning repo...')
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
os.chdir(PROJECT_DIR)
print('Force-updating to origin/main...')
!git fetch --quiet origin
!git reset --hard --quiet origin/main

vec_dir = os.path.join(PROJECT_DIR, 'VECTOR')
sys.path.insert(0, vec_dir)

# Triton powers BOTH fused scans (SSM + gated delta). Install if absent.
try:
    import triton
except ImportError:
    !pip install -q triton
    import triton
sys.modules.pop('triton_scan', None)
import triton_scan
print(f'Triton {triton.__version__} | triton_scan HAS_TRITON={triton_scan.HAS_TRITON}')
assert triton_scan.HAS_TRITON, 'Triton unavailable -- fused scans will fall back to JIT'
print('Environment OK')


## Retrieval A/B: does content-based recall close the GPT gap?

**Why:** Stream is O(n), but a fixed `d_state` recurrence cannot do
content-based recall — GPT beats it at matched small scale. This cell tests
two answers inside one training loop, in the same socket (last blocks swapped):

- **StreamR** — multi-pathway *bounded attention* retrieval: dense window +
  strided far window + content-derived segment memory + global tokens, with
  per-head relative biases and a learned per-head fusion gate. All pathways
  are O(n); enabling them is the v2 default.
- **StreamD** — *pure recurrence*: each head owns an associative matrix
  updated by a gated delta rule (erase + write, read-before-write => strictly
  causal). No attention anywhere, fixed-size state, λ≈0.9 decay gates.

**Self-contained:** `model.py` + `delta_scan.py` are embedded in the cell and
ALWAYS rewritten, so the cell works even if the clone above is stale.
Correctness suites run first; both fused Triton scans are verified against
references on-GPU before being enabled (fallback keeps everything working).

**Verdicts print automatically** at the end: bounded-retrieval-helps
(StreamR < Stream), delta-helps (StreamD < Stream), content-addressed-beats-
bounded (StreamD < StreamR), and gap-to-GPT closed.


In [ ]:
# @title 2. Retrieval A/B: Stream vs StreamR(v2) vs StreamD vs GPT (T=4096 & T=16384)
# A/B: does a sparse-retrieval layer close the GPT gap?
# Stream vs StreamR(v2) vs StreamD vs GPT (3L, RoPE) - same bytes,
# same batches, same steps, matched scale. Legs: T=4096 (quality) and T=16384
# (long context: val loss + step time). All three models see IDENTICAL data.
# Caveat: the GPT T=16384 leg is O(n^2) attention and will be the slow part -
# that is exactly the point (Stream/StreamR are O(n)).
# SELF-CONTAINED: model.py + delta_scan.py sources are embedded below and ALWAYS
# rewritten, so this cell runs even if the Colab clone is stale (no git pull).
import os, sys, time, math, importlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

_MODEL_SRC = r'''"""
Stream: Continuous Byte-Level SSM
Token-free, position-free, O(n) language model.
Predicts next N bytes directly from raw bytes — no tokenizer, no PE, no gate, no MoE.

Architecture:
- Byte embedding (256 → D) — the only "vocabulary"
- Stacked SSM blocks — recurrence = position by construction
- Optional sparse-retrieval blocks (windowed attention + global tokens) for
  content-based recall that a fixed d_state recurrence cannot do, while staying
  O(n) memory (window, not full attention). Off by default (n_retrieval=0).
- Multi-byte head: predict next N bytes per position
- Single loss: next-byte CE summed over N future predictions
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from dataclasses import dataclass
from typing import Optional, Tuple, List, TYPE_CHECKING

try:
    from delta_scan import delta_scan_fused
except Exception:            # delta_scan optional; eager loop is the fallback
    delta_scan_fused = None


# -----------------------------------------------------------------------------
# SSM scan: JIT-compiled sequential recurrence.
# On CPU the sequential loop is optimal (Blelloch tree scan adds overhead from
# non-contiguous access). JIT eliminates Python loop overhead.
# -----------------------------------------------------------------------------

# ── JIT-compiled forward/backward scan loops ───────────────────────
# TorchScript fuses the per-step elementwise ops into a single CUDA kernel,
# eliminating the O(T) kernel-launch overhead from the Python loop.

@torch.jit.script
def _ssm_fwd(a_vec: torch.Tensor, b_vec: torch.Tensor, T_s: int) -> torch.Tensor:
    Bs, _, Hc, Nc = a_vec.shape
    h = torch.zeros(Bs, Hc, Nc, device=a_vec.device)
    out = torch.empty(Bs, T_s, Hc, Nc, device=a_vec.device)
    for t in range(T_s):
        h = h * a_vec[:, t] + b_vec[:, t]
        out[:, t] = h
    return out

@torch.jit.script
def _ssm_bwd(grad_output: torch.Tensor, a_vec: torch.Tensor,
             out: torch.Tensor) -> List[torch.Tensor]:
    Bs, T_s, Hc, Nc = a_vec.shape
    grad_a = torch.zeros_like(a_vec); grad_b = torch.zeros_like(a_vec)
    dh = torch.zeros(Bs, Hc, Nc, device=a_vec.device)
    for t in range(T_s - 1, -1, -1):
        dh_total = grad_output[:, t] + dh
        h_prev = out[:, t - 1] if t > 0 else torch.zeros(Bs, Hc, Nc, device=a_vec.device)
        grad_b[:, t] = dh_total; grad_a[:, t] = dh_total * h_prev
        dh = dh_total * a_vec[:, t]
    return [grad_a, grad_b]

class SSMScanFn(torch.autograd.Function):
    """
    Custom autograd Function wrapping JIT-compiled scan kernels.
    The JIT-compiled forward/backward loops are fused into single CUDA
    kernels, eliminating per-step Python overhead and most kernel-launch
    overhead. The custom backward avoids building the full O(T) autograd
    graph that PyTorch would construct from the loop.
    """
    @staticmethod
    def forward(ctx, a_vec, b_vec, T_s):
        out = _ssm_fwd(a_vec, b_vec, T_s)
        ctx.save_for_backward(a_vec, out)
        ctx.T_s = T_s
        return out

    @staticmethod
    def backward(ctx, grad_output):
        a_vec, out = ctx.saved_tensors
        grad_a, grad_b = _ssm_bwd(grad_output, a_vec, out)
        return grad_a, grad_b, None


def _ssm_scan(a_vec, b_vec, T):
    """Wrapper that calls SSMScanFn.apply."""
    return SSMScanFn.apply(a_vec, b_vec, T)


def parallel_ssm_scan(u: torch.Tensor, dt: torch.Tensor,
                      A: torch.Tensor, B: torch.Tensor, C: torch.Tensor,
                      D: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    SSM scan: sequential recurrence h_{t+1} = a_t · h_t + b_t, h_0 = 0,
    with a_t and b_t being functions of the input.

    Uses a JIT-compiled loop over T to eliminate Python overhead.
    On CPU there is no O(log T) parallel advantage (tree scan adds non-contiguous
    access cost), but the JIT avoids O(T) Python-level iteration cost.

    Args:
      u:  (B, T, H)    input
      dt: (B, T, H)    step sizes
      A:  (H, N)       state matrix (negative = -exp(A_log))
      B:  (B, T, N)    input projection
      C:  (B, T, N)    output projection
      D:  (H,)         skip connection

    Returns:
      y:     (B, T, H)  output
      state: (B, H, N)  final hidden state (detached)
    """
    Bs, T, H = u.shape
    N = A.shape[-1]

    # Precompute transition a_t and input b_t
    a_vec = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))  # (B, T, H, N)
    b_vec = dt.unsqueeze(-1) * B.unsqueeze(2) * u.unsqueeze(-1)        # (B, T, H, N)

    # Custom autograd scan — h[t] = state after processing input t
    h = _ssm_scan(a_vec, b_vec, T)  # (B, T, H, N)

    # Output: y[t] = (h[t] · C[t]).sum(-1) + D · u[t]
    y = (h * C.unsqueeze(2)).sum(-1) + D * u

    return y, (h[:, -1].detach(), None)


# -----------------------------------------------------------------------------
# SSM Block: selective state space (Mamba-style)
# -----------------------------------------------------------------------------
class SSMBlock(nn.Module):
    def __init__(self, n_embd: int, ssm_d_state: int = 16,
                 ssm_d_conv: int = 4, ssm_expand: int = 2, bias: bool = False):
        super().__init__()
        self.n_embd = n_embd
        self.ssm_d_state = ssm_d_state
        self.ssm_d_conv = ssm_d_conv
        hidden = n_embd * ssm_expand

        self.in_proj = nn.Linear(n_embd, hidden * 2, bias=bias)
        self.conv1d = nn.Conv1d(hidden, hidden, kernel_size=ssm_d_conv,
                                padding=ssm_d_conv - 1, groups=hidden, bias=bias)
        self.act = nn.SiLU()
        self.x_proj = nn.Linear(hidden, ssm_d_state * 2, bias=bias)
        self.dt_proj = nn.Linear(hidden, hidden, bias=True)

        self.A_log = nn.Parameter(torch.zeros(hidden, ssm_d_state))
        self.D = nn.Parameter(torch.randn(hidden))
        self.out_proj = nn.Linear(hidden, n_embd, bias=bias)
        self.ln = nn.LayerNorm(n_embd)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, 'SSMBlockState']:
        B, T, D = x.shape
        H = self.n_embd * (D // self.n_embd) if D != self.n_embd else self.n_embd * 2

        x_proj = self.in_proj(x)
        x_main, gate = x_proj.chunk(2, dim=-1)
        x_main = self.act(x_main)
        gate = torch.sigmoid(gate)

        x_conv = self.conv1d(x_main.transpose(1, 2))[..., :T].transpose(1, 2)
        x_conv = self.act(x_conv)

        dt = F.softplus(self.dt_proj(x_conv))
        B_param, C_param = self.x_proj(x_conv).chunk(2, dim=-1)
        A = -torch.exp(self.A_log.float())

        y, ssm_state = self._ssm_scan(x_conv, dt, A, B_param, C_param)
        # The historical scan wrappers returned `(state, aux)`; accept that
        # ABI while the stateful inference contract stores the tensor itself.
        if isinstance(ssm_state, tuple):
            ssm_state = ssm_state[0]
        y = y * gate
        out = self.out_proj(y)
        history_len = self.ssm_d_conv - 1
        conv_history = (x_main[:, -history_len:].detach() if history_len else
                        x_main[:, :0].detach())
        return self.ln(out + x), SSMBlockState(ssm=ssm_state, conv=conv_history)

    def _ssm_scan(self, u, dt, A, B, C):
        return parallel_ssm_scan(u, dt, A, B, C, self.D)

    @torch.no_grad()
    def step(self, x: torch.Tensor, state: Optional['SSMBlockState'] = None
             ) -> Tuple[torch.Tensor, 'SSMBlockState']:
        """Process exactly one byte position while carrying only recurrent state.

        This is intentionally separate from the training scan: it is the
        correctness reference for the eventual fused decode kernel.  `x` has
        shape `(batch, n_embd)` and the returned state is detached, making its
        memory independent of generated length.
        """
        if x.ndim != 2:
            raise ValueError(f"SSMBlock.step expects (B, D), got {tuple(x.shape)}")
        B, D = x.shape
        x_proj = self.in_proj(x)
        x_main, gate = x_proj.chunk(2, dim=-1)
        x_main = self.act(x_main)
        gate = torch.sigmoid(gate)

        history_len = self.ssm_d_conv - 1
        if state is None:
            conv_history = x_main.new_zeros(B, history_len, x_main.shape[-1])
            ssm_state = x_main.new_zeros(B, x_main.shape[-1], self.ssm_d_state)
        else:
            conv_history, ssm_state = state.conv, state.ssm
            expected_history = (B, history_len, x_main.shape[-1])
            expected_ssm = (B, x_main.shape[-1], self.ssm_d_state)
            if tuple(conv_history.shape) != expected_history or tuple(ssm_state.shape) != expected_ssm:
                raise ValueError("SSM state shape does not match this block or batch")

        conv_input = torch.cat((conv_history, x_main.unsqueeze(1)), dim=1)
        x_conv = F.conv1d(conv_input.transpose(1, 2), self.conv1d.weight,
                          self.conv1d.bias, groups=self.conv1d.groups).squeeze(-1)
        x_conv = self.act(x_conv)
        dt = F.softplus(self.dt_proj(x_conv))
        B_param, C_param = self.x_proj(x_conv).chunk(2, dim=-1)
        A = -torch.exp(self.A_log.float()).to(dtype=x.dtype)
        a = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0))
        b = dt.unsqueeze(-1) * B_param.unsqueeze(1) * x_conv.unsqueeze(-1)
        next_ssm = ssm_state * a + b
        y = (next_ssm * C_param.unsqueeze(1)).sum(-1) + self.D.to(x.dtype) * x_conv
        out = self.ln(self.out_proj(y * gate) + x)
        next_history = (conv_input[:, -history_len:].detach() if history_len else
                        conv_input[:, :0].detach())
        return out, SSMBlockState(ssm=next_ssm.detach(), conv=next_history)


@dataclass
class SSMBlockState:
    """Persistent state of one selective-SSM layer during inference."""
    ssm: torch.Tensor
    conv: torch.Tensor


# -----------------------------------------------------------------------------
# Sparse Retrieval Block v2: multi-pathway content recall, O(n) memory.
# -----------------------------------------------------------------------------
def _causal_window(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                   w: int, rel_bias: torch.Tensor, per_head: bool,
                   chunk: Optional[int] = None) -> torch.Tensor:
    """Exact causal window attention with bounded peak memory.

    Same math as the full `unfold` formulation, but key/value windows are
    materialized per query-chunk so the (B, T, D, w) intermediate never
    exists in full (peak O(B*C*D*w) instead of O(B*T*D*w) -- at leg B this
    is the difference between ~4.3 GB of temporaries and ~67 MB).

    q: (B, nh, T, hd); k, v: raw (B, T, D); rel_bias shared (2w-1,) or
    per-head (nh, 2w-1). Returns (B, nh, T, hd). When C >= T this reduces
    to exactly one full-sequence chunk (identical op sequence as before).
    """
    B, nh, T, hd = q.shape
    D = k.shape[-1]
    if chunk is None:
        # keep each chunk's window temporaries <= ~2**24 elements
        chunk = max(64, min(T, int((2 ** 24) // max(1, B * D * w))))
    chunk = min(chunk, T)
    k_pad = F.pad(k, (0, 0, w - 1, 0))   # left-pad: query t sees keys t-w+1..t
    v_pad = F.pad(v, (0, 0, w - 1, 0))
    dist = (w - 1) - torch.arange(w, device=q.device)
    if per_head:
        bias = rel_bias[:, dist].view(1, nh, 1, w)
    else:
        bias = rel_bias[dist].view(1, 1, 1, w)
    outs = []
    for s0 in range(0, T, chunk):
        s1 = min(s0 + chunk, T)
        L = s1 - s0
        kw = k_pad[:, s0:s1 + w - 1].unfold(1, w, 1)          # (B, L, D, w)
        vw = v_pad[:, s0:s1 + w - 1].unfold(1, w, 1)
        kw = kw.view(B, L, nh, hd, w).transpose(1, 2)         # (B, nh, L, hd, w)
        vw = vw.view(B, L, nh, hd, w).transpose(1, 2)
        lw = torch.einsum('bhtd,bhtdw->bhtw', q[:, :, s0:s1], kw) * (hd ** -0.5)
        lw = lw + bias
        att_w = torch.softmax(lw, dim=-1)
        outs.append(torch.einsum('bhtw,bhtdw->bhtd', att_w, vw))
    return torch.cat(outs, dim=2)


class RetrievalBlock(nn.Module):
    """
    Gives the SSM backbone content-based recall which a fixed d_state recurrence
    cannot do, while keeping O(n) memory and position-free relative biases.

    Pathways (all causal, all translation-invariant, no absolute PE):
      1. Dense window  — exact recall over the last `window` positions (v1).
      2. Strided window — exact recall at a stride beyond the dense window
         (offsets w+1, w+1+s, w+1+2s, ...), reaching ~w + slots·s bytes back.
      3. Segment memory — content-derived: the sequence is split into
         `mem_seg`-byte segments and each segment's k/v are attention-pooled by a
         learned per-head query into one (k, v) pair; every position attends to
         the `mem_slots` most recent segments before its own. Unlike v1's static
         sink tokens, these keys/values ARE the pooled content, so distant bytes
         are actually retrievable.
      4. Global tokens  — a small set of static learned sinks (anchors).

    Fusion: each head learns per-pathway weights (softmax over active pathways),
    so different heads can specialize on local vs. far recall — the gating idea
    from the design doc, at zero per-position cost.

    When all v2 features are OFF (per_head_bias=False, stride=0, mem_slots=0,
    gated=False) this block reproduces v1 EXACTLY (shared rel_bias, joint
    softmax over [global, window]), so the pilot and CELL 13 numbers remain
    reproducible. Enabling any feature switches to the pathway-fusion path.

    Output: post-norm residual like SSMBlock (ln(proj(y) + x), None).
    """
    def __init__(self, n_embd: int, n_head: int = 4, window: int = 128,
                 n_global: int = 16, bias: bool = False,
                 per_head_bias: bool = False,
                 stride: int = 0, stride_slots: int = 16,
                 mem_slots: int = 0, mem_seg: int = 32,
                 gated: bool = False):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_embd = n_embd
        self.n_head = n_head
        self.window = window
        self.n_global = n_global
        self.head_dim = n_embd // n_head
        self.per_head_bias = per_head_bias
        self.stride = stride
        self.stride_slots = stride_slots
        self.mem_slots = mem_slots
        self.mem_seg = mem_seg
        self.gated = gated

        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=bias)
        self.proj = nn.Linear(n_embd, n_embd, bias=bias)
        if n_global > 0:
            self.g_k = nn.Parameter(torch.randn(n_global, n_head, self.head_dim) * 0.02)
            self.g_v = nn.Parameter(torch.randn(n_global, n_head, self.head_dim) * 0.02)
        else:
            self.register_buffer('g_k', torch.zeros(0, n_head, self.head_dim))
            self.register_buffer('g_v', torch.zeros(0, n_head, self.head_dim))

        if per_head_bias:
            self.rel_bias = nn.Parameter(torch.zeros(n_head, 2 * window - 1))
        else:
            self.rel_bias = nn.Parameter(torch.zeros(2 * window - 1))

        if stride > 0:
            if per_head_bias:
                self.stride_bias = nn.Parameter(torch.zeros(n_head, stride_slots))
            else:
                self.stride_bias = nn.Parameter(torch.zeros(stride_slots))
        if mem_slots > 0:
            self.seg_q = nn.Parameter(torch.randn(n_head, self.head_dim) * 0.02)
            if per_head_bias:
                self.mem_bias = nn.Parameter(torch.zeros(n_head, mem_slots))
            else:
                self.mem_bias = nn.Parameter(torch.zeros(mem_slots))

        if gated:
            n_paths = (1 + (1 if stride > 0 else 0)
                       + (1 if mem_slots > 0 else 0)
                       + (1 if n_global > 0 else 0))
            self.path_logits = nn.Parameter(torch.zeros(n_paths, n_head))
        self.ln = nn.LayerNorm(n_embd)

    @staticmethod
    def _masked_softmax(logits: torch.Tensor, mask: torch.Tensor, dim: int) -> torch.Tensor:
        """Softmax over `dim` with binary mask; invalid slots get ~0 weight (never NaN)."""
        fill = -1e4 if logits.dtype == torch.float16 else -1e9
        l = torch.where(mask, logits, torch.full_like(logits, fill))
        a = torch.softmax(l, dim=dim)
        a = a * mask
        return a / (a.sum(dim=dim, keepdim=True) + 1e-12)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, D = x.shape
        nh, hd, w, ng = self.n_head, self.head_dim, self.window, self.n_global

        q, k, v = self.qkv(x).chunk(3, dim=-1)            # (B, T, D)
        q = q.view(B, T, nh, hd).transpose(1, 2)          # (B, nh, T, hd)

        is_v2 = (self.per_head_bias or self.stride > 0
                 or self.mem_slots > 0 or self.gated)
        if not is_v2:
            # ---- exact v1 path (byte-identical to the original block) ----
            k_pad = F.pad(k, (0, 0, w - 1, 0))
            v_pad = F.pad(v, (0, 0, w - 1, 0))
            k_win = k_pad.unfold(1, w, 1).view(B, T, nh, hd, w).transpose(1, 2)
            v_win = v_pad.unfold(1, w, 1).view(B, T, nh, hd, w).transpose(1, 2)
            lw = torch.einsum('bhtd,bhtdw->bhtw', q, k_win) * (hd ** -0.5)
            dist = (w - 1) - torch.arange(w, device=x.device)
            lw = lw + self.rel_bias[dist].view(1, 1, 1, w)
            lg = torch.einsum('bhtd,ghd->bhtg', q, self.g_k) * (hd ** -0.5)
            att = torch.softmax(torch.cat([lg, lw], dim=-1), dim=-1)
            ow = torch.einsum('bhtw,bhtdw->bhtd', att[..., ng:], v_win)
            og = torch.einsum('bhtg,ghd->bhtd', att[..., :ng], self.g_v)
            y = (ow + og).transpose(1, 2).reshape(B, T, D)
            return self.ln(self.proj(y) + x), None

        k_h = k.view(B, T, nh, hd)
        v_h = v.view(B, T, nh, hd)
        outs = []

        # ---- pathway 1: dense window (v1 mechanism, optional per-head bias) ----
        outs.append(_causal_window(q, k, v, w, self.rel_bias, self.per_head_bias))

        # ---- pathway 2: strided far window (exact recall at distance) ----
        if self.stride > 0:
            Ks = self.stride_slots
            offsets = w + 1 + torch.arange(Ks, device=x.device) * self.stride   # (Ks,)
            pos_idx = torch.arange(T, device=x.device).unsqueeze(1) - offsets.unsqueeze(0)  # (T, Ks)
            valid_s = pos_idx >= 0
            idx_s = pos_idx.clamp(min=0)   # (T, Ks)
            k_str = k_h[:, idx_s, :, :]    # (B, T, Ks, nh, hd) advanced index on dim 1
            v_str = v_h[:, idx_s, :, :]
            k_str = k_str.permute(0, 3, 1, 2, 4).contiguous()   # (B, nh, T, Ks, hd)
            v_str = v_str.permute(0, 3, 1, 2, 4).contiguous()
            ls = torch.einsum('bhtd,bhtsd->bhts', q, k_str) * (hd ** -0.5)
            if self.per_head_bias:
                ls = ls + self.stride_bias.unsqueeze(0).unsqueeze(2)
            else:
                ls = ls + self.stride_bias.view(1, 1, 1, Ks)
            att_s = self._masked_softmax(ls, valid_s.unsqueeze(0).unsqueeze(0), dim=-1)   # (B, nh, T, Ks)
            os_ = torch.einsum('bhts,bhtsd->bhtd', att_s, v_str)
            outs.append(os_)

        # ---- pathway 3: content-derived segment memory ----
        if self.mem_slots > 0:
            ms, Km = self.mem_seg, self.mem_slots
            n_seg = (T + ms - 1) // ms
            padT = n_seg * ms - T
            # pad T on the left (positions before 0) with zeros; pooling rows for
            # padded tail tokens are masked out so no dummy content leaks in.
            k_seg = F.pad(k_h, (0, 0, 0, 0, padT, 0))      # (B, n_seg*ms, nh, hd)
            v_seg = F.pad(v_h, (0, 0, 0, 0, padT, 0))
            k_seg = k_seg.view(B, n_seg, ms, nh, hd)
            v_seg = v_seg.view(B, n_seg, ms, nh, hd)
            # learned per-head pooling query → content-derived summary per segment
            sp = torch.einsum('bsmhd,hd->bsmh', k_seg, self.seg_q) * (hd ** -0.5)
            pos_tok = torch.arange(n_seg * ms, device=x.device).view(1, n_seg, ms)
            seg_valid = pos_tok < T          # (1, n_seg, ms)
            att_pool = self._masked_softmax(sp, seg_valid.unsqueeze(-1).expand(B, n_seg, ms, nh), dim=2)
            mem_k = torch.einsum('bsmh,bsmhd->bshd', att_pool, k_seg)   # (B, n_seg, nh, hd)
            mem_v = torch.einsum('bsmh,bsmhd->bshd', att_pool, v_seg)
            # each position attends to the last Km segments strictly before its own
            seg_idx = (torch.arange(T, device=x.device) // ms)           # (T,)
            m_start = (seg_idx - Km).clamp(min=0)                        # (T,)
            m_idx = m_start.unsqueeze(1) + torch.arange(Km, device=x.device).unsqueeze(0)  # (T, Km)
            valid_m = m_idx < seg_idx.unsqueeze(1)                       # segment must precede own
            m_idx_c = m_idx.clamp(min=0, max=n_seg - 1)                  # (T, Km)
            gk = mem_k[:, m_idx_c].permute(0, 3, 1, 2, 4).contiguous()      # (B, nh, T, Km, hd)
            gv = mem_v[:, m_idx_c].permute(0, 3, 1, 2, 4).contiguous()
            lm = torch.einsum('bhtd,bhtkd->bhtk', q, gk) * (hd ** -0.5)
            rel_dist = seg_idx.unsqueeze(1) - m_idx                      # 1..Km (larger = older)
            if self.per_head_bias:
                lm = lm + self.mem_bias[:, (rel_dist - 1).clamp(min=0)].unsqueeze(0)
            else:
                lm = lm + self.mem_bias[(rel_dist - 1).clamp(min=0)].view(1, 1, T, Km)
            att_m = self._masked_softmax(lm, valid_m.unsqueeze(0).unsqueeze(0), dim=-1)
            om = torch.einsum('bhtk,bhtkd->bhtd', att_m, gv)
            outs.append(om)

        # ---- pathway 4: static global tokens ----
        if ng > 0:
            lg = torch.einsum('bhtd,ghd->bhtg', q, self.g_k) * (hd ** -0.5)
            att_g = torch.softmax(lg, dim=-1)
            og = torch.einsum('bhtg,ghd->bhtd', att_g, self.g_v)
            outs.append(og)

        # ---- fusion: learned per-head pathway weights (gating) ----
        if self.gated:
            wts = torch.softmax(self.path_logits, dim=0)   # (n_paths, nh)
            y = sum(wts[p].view(1, nh, 1, 1) * o for p, o in enumerate(outs))
        else:
            y = sum(outs) if len(outs) > 1 else outs[0]
        y = y.transpose(1, 2).reshape(B, T, D)
        return self.ln(self.proj(y) + x), None


# -----------------------------------------------------------------------------
# Gated Delta Memory Block: pure-recurrence content-addressed retrieval.
# -----------------------------------------------------------------------------
class DeltaMemoryBlock(nn.Module):
    """
    Content-addressed memory via a per-head associative matrix updated with the
    delta rule — the pure-recurrence alternative to RetrievalBlock's bounded
    attention. No tokens, no positions, no PE, no O(T^2): the state is exactly
    nh small matrices W_h in R^(hd x hd), and every token does O(hd^2) work.

      read : o_t   = W_{t-1} @ q_t                       (content-addressed)
      write: pred  = W_{t-1} @ k_t                       (predict stored value)
             W_t   = lam_t * W_{t-1} + beta_t * (v_t - pred) otimes k_t

    lam_t (decay) and beta_t (write gain) are learned per head and input-driven:
    a head can act as a persistent associative store (high lam, low beta) or a
    volatile short-term buffer (low lam, high beta). k is RMSNorm-normalized per
    head (unit-length keys bound interference and keep the matrix stable in
    fp16). Reads use the state BEFORE the current token's write, so recall is
    strictly of the past — causal by construction, like the SSM.

    With delta_window > 0 the block adds an exact-recent dense window pathway
    (causal unfold, like RetrievalBlock) and fuses it with the matrix via a
    learned per-head softmax gate; delta_window=0 is pure recurrence.

    Output: post-norm residual like the other blocks (ln(proj(y) + x), None).
    """
    def __init__(self, n_embd: int, n_head: int = 4, window: int = 0,
                 bias: bool = False, per_head_bias: bool = True,
                 lam_init: float = 2.2, beta_init: float = 0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_embd = n_embd
        self.n_head = n_head
        self.window = window
        self.head_dim = n_embd // n_head
        self.per_head_bias = per_head_bias
        self.lam_init = lam_init
        self.beta_init = beta_init
        self._use_fused = False   # set True by delta_scan.enable_delta_triton

        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=bias)
        # per-head gate logits: [lambda_raw, beta_raw] -> (B, T, 2*nh)
        self.gate_logits = nn.Linear(n_embd, 2 * n_head, bias=True)
        self.reset_gate_bias()
        self.k_norm = nn.LayerNorm(self.head_dim, bias=False)   # per-head RMS-ish
        self.proj = nn.Linear(n_embd, n_embd, bias=bias)
        self.ln = nn.LayerNorm(n_embd)

        if window > 0:
            if per_head_bias:
                self.rel_bias = nn.Parameter(torch.zeros(n_head, 2 * window - 1))
            else:
                self.rel_bias = nn.Parameter(torch.zeros(2 * window - 1))
            self.path_logits = nn.Parameter(torch.zeros(2, n_head))

    def reset_gate_bias(self):
        """Zero the gate weights and pin the lambda/beta logit biases.

        MUST be (re-)applied after any blanket nn.Linear re-initialization
        (e.g. Stream._init_weights via self.apply), which would otherwise
        wipe the pinned biases: sigmoid(0)=0.5 decay instead of ~0.90."""
        with torch.no_grad():
            self.gate_logits.weight.zero_()
            self.gate_logits.bias.zero_()
            nh = self.n_head
            self.gate_logits.bias[0:nh] = self.lam_init
            self.gate_logits.bias[nh:] = self.beta_init

    def _delta_scan(self, S, q, k, v, lam, beta):
        """Dispatch: fused Triton kernel on CUDA when enabled, else eager loop."""
        if self._use_fused and delta_scan_fused is not None and q.is_cuda:
            return delta_scan_fused(q, k, v, lam, beta)
        return self._delta_scan_eager(S, q, k, v, lam, beta)

    @torch.jit.ignore
    def _delta_scan_eager(self, S, q, k, v, lam, beta):
        """Sequential delta recurrence over T, vectorized over (B, nh).
        S:   (B, nh, hd, hd)  state (zeros at entry)
        q/k/v: (B, T, nh, hd)
        lam/beta: (B, T, nh)
        returns o (B, T, nh, hd), final S (detached)
        """
        B, T, nh, hd = q.shape
        outs = torch.empty_like(q)
        lam = lam.unsqueeze(-1).unsqueeze(-1)   # (B, T, nh, 1, 1)
        beta = beta.unsqueeze(-1).unsqueeze(-1)  # (B, T, nh, 1, 1)
        for t in range(T):
            kt = k[:, t]                 # (B, nh, hd)
            qt = q[:, t]
            vt = v[:, t]
            # read-before-write: strictly past recall
            Sk = torch.matmul(S, kt.unsqueeze(-1)).squeeze(-1)      # (B, nh, hd)
            o = torch.matmul(S, qt.unsqueeze(-1)).squeeze(-1)       # (B, nh, hd)
            outs[:, t] = o
            # delta write: erase along kt, add v-prediction correction
            err = vt - Sk                                          # (B, nh, hd)
            S = lam[:, t] * S + beta[:, t] * kt.unsqueeze(-1) * err.unsqueeze(-2)
        return outs, S.detach()

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, D = x.shape
        nh, hd, w = self.n_head, self.head_dim, self.window

        q, k, v = self.qkv(x).chunk(3, dim=-1)                 # (B, T, D)
        k_raw = k
        # normalize keys to unit-ish length per head (bounded interference, fp16-safe)
        k = self.k_norm(k.view(B, T, nh, hd)).view(B, T, nh, hd)
        qh = q.view(B, T, nh, hd)
        vh = v.view(B, T, nh, hd)

        g = torch.sigmoid(self.gate_logits(x))                 # (B, T, 2*nh)
        lam, beta = g[..., :nh], g[..., nh:]                   # (B, T, nh)

        S0 = q.new_zeros(B, nh, hd, hd)
        o_d, _ = self._delta_scan(S0, qh, k, vh, lam, beta)    # (B, T, nh, hd)

        if w > 0:
            # hybrid pathway: exact-recent window, fused with the associative
            # response by a learned per-head gate over [matrix, window].
            o_w = _causal_window(qh.transpose(1, 2), k_raw, v, w,
                                 self.rel_bias, self.per_head_bias)  # (B, nh, T, hd)

            o_d = o_d.transpose(1, 2)                          # (B, nh, T, hd)
            wts = torch.softmax(self.path_logits, dim=0)       # (2, nh)
            o = (wts[0].view(1, nh, 1, 1) * o_d
                 + wts[1].view(1, nh, 1, 1) * o_w)             # (B, nh, T, hd)
            o = o.transpose(1, 2).reshape(B, T, D)
        else:
            o = o_d.view(B, T, D)

        return self.ln(self.proj(o) + x), None
@dataclass
class StreamConfig:
    vocab_size: int = 256
    n_embd: int = 256
    n_layer: int = 6
    ssm_d_state: int = 16
    n_predict: int = 4
    block_size: int = 1024
    dropout: float = 0.0
    bias: bool = False
    # Sparse retrieval (off by default). Retrieval blocks replace the LAST
    # n_retrieval SSM blocks so the head directly sees retrieved content.
    n_retrieval: int = 0
    n_attn_head: int = 4
    window_size: int = 128
    n_global: int = 16
    # RetrievalBlock v2 upgrades (all off ⇒ exact v1 behavior).
    per_head_bias: bool = False       # per-head rel bias instead of shared
    retr_stride: int = 0              # >0: strided far window gap
    retr_stride_slots: int = 16       # how many strided slots per query
    retr_mem_slots: int = 0           # >0: content-derived segment memory (count)
    retr_mem_seg: int = 32            # bytes per memory segment
    retr_gated: bool = False          # learned per-head pathway fusion
    # Gated delta memory (off by default). Delta blocks also replace the LAST
    # n_delta SSM blocks (after any retrieval blocks); pure recurrence + O(hd^2).
    n_delta: int = 0
    delta_head: int = 4
    delta_window: int = 0             # >0: hybrid add exact-recent window pathway
    delta_lam_init: float = 2.2       # gate logit bias -> lam ~= sigmoid(2.2) ~ 0.90
    delta_beta_init: float = 0.0      # gate logit bias -> beta ~= 0.50
    activation_checkpointing: bool = False  # trade recompute for T4 VRAM


@dataclass
class StreamState:
    """Constant-size inference state for a pure Stream stack."""
    blocks: List[SSMBlockState]


class Stream(nn.Module):
    def __init__(self, config: StreamConfig):
        super().__init__()
        self.config = config

        self.byte_embed = nn.Embedding(config.vocab_size, config.n_embd)

        n_ssm = max(0, config.n_layer - config.n_retrieval - config.n_delta)
        self.blocks = nn.ModuleList(
            [SSMBlock(config.n_embd, ssm_d_state=config.ssm_d_state, bias=config.bias)
             for _ in range(n_ssm)]
            + [RetrievalBlock(config.n_embd, n_head=config.n_attn_head,
                              window=config.window_size, n_global=config.n_global,
                              bias=config.bias,
                              per_head_bias=config.per_head_bias,
                              stride=config.retr_stride,
                              stride_slots=config.retr_stride_slots,
                              mem_slots=config.retr_mem_slots,
                              mem_seg=config.retr_mem_seg,
                              gated=config.retr_gated)
               for _ in range(config.n_retrieval)]
            + [DeltaMemoryBlock(config.n_embd, n_head=config.delta_head,
                                window=config.delta_window,
                                bias=config.bias,
                                lam_init=config.delta_lam_init,
                                beta_init=config.delta_beta_init)
               for _ in range(config.n_delta)]
        )
        self.ln_f = nn.LayerNorm(config.n_embd)

        self.head = nn.Linear(
            config.n_embd,
            config.n_predict * config.vocab_size,
            bias=False
        )

        self.apply(self._init_weights)
        # self.apply re-initialized every nn.Linear above, which wipes the
        # pinned lambda/beta gate biases of DeltaMemoryBlock -- re-pin them.
        for _m in self.modules():
            if isinstance(_m, DeltaMemoryBlock):
                _m.reset_gate_bias()
        for pn, p in self.named_parameters():
            # out_proj / c_proj / retrieval proj get GPT-2-style residual scaling.
            # '.proj.weight' matches only the retrieval block's proj (dt_proj and
            # out_proj have an underscore before 'proj', so they don't match).
            if (pn.endswith('out_proj.weight') or pn.endswith('c_proj.weight')
                    or pn.endswith('.proj.weight')):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

        print(f"Stream parameters: {self.get_num_params() / 1e6:.2f}M")

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor,
                targets: Optional[torch.Tensor] = None,
                return_logits: bool = False,
                iter_num: int = 0,
                return_state: bool = False):
        B, T = idx.shape
        assert T <= self.config.block_size

        x = self.byte_embed(idx)

        block_states = []
        for block in self.blocks:
            if self.training and self.config.activation_checkpointing and not return_state:
                # Recompute each block in backward instead of retaining its
                # activations. `use_reentrant=False` is robust with the custom
                # SSM autograd function and does not change model numerics.
                x = checkpoint(lambda h, layer=block: layer(h)[0], x,
                               use_reentrant=False)
                block_state = None
            else:
                x, block_state = block(x)
            block_states.append(block_state)

        x = self.ln_f(x)
        logits = self.head(x)

        if targets is not None:
            loss = self._compute_loss(logits, targets)
        else:
            loss = None

        state = StreamState(blocks=block_states) if return_state else None
        if return_state:
            return logits, loss, state
        return logits, loss

    def _compute_loss(self, logits, targets):
        B, T, _ = logits.shape
        np = self.config.n_predict
        vs = self.config.vocab_size
        logits = logits.view(B, T, np, vs)

        loss = 0.0
        for k in range(np):
            loss = loss + F.cross_entropy(
                logits[:, :T - k, k].reshape(-1, vs),
                targets[:, k:].reshape(-1),
                ignore_index=-1
            )
        return loss / np

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        import inspect
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        print(f"using fused AdamW: {use_fused}")
        return optimizer

    def _require_streaming_blocks(self):
        unsupported = [type(block).__name__ for block in self.blocks
                       if not isinstance(block, SSMBlock)]
        if unsupported:
            joined = ', '.join(sorted(set(unsupported)))
            raise NotImplementedError(
                f"Stateful streaming is currently defined only for pure SSM Stream; "
                f"{joined} needs a verified carried-state kernel first.")

    @torch.no_grad()
    def prefill(self, idx: torch.Tensor) -> Tuple[torch.Tensor, StreamState]:
        """Encode a non-empty byte prefix and return next-byte logits and state."""
        self._require_streaming_blocks()
        if idx.ndim != 2 or idx.shape[1] == 0:
            raise ValueError("prefill expects a non-empty (B, T) byte tensor")
        if idx.shape[1] > self.config.block_size:
            raise ValueError("prefix exceeds training block_size; chunk prefill explicitly")
        logits, _, state = self(idx, return_state=True)
        return logits[:, -1, :self.config.vocab_size], state

    @torch.no_grad()
    def step(self, idx: torch.Tensor, state: StreamState) -> Tuple[torch.Tensor, StreamState]:
        """Consume one byte per batch item and return logits for its successor."""
        self._require_streaming_blocks()
        if idx.ndim == 2 and idx.shape[1] == 1:
            idx = idx[:, 0]
        if idx.ndim != 1:
            raise ValueError("step expects (B,) or (B, 1) byte ids")
        if len(state.blocks) != len(self.blocks):
            raise ValueError("StreamState belongs to a different model")
        x = self.byte_embed(idx)
        next_states = []
        for block, block_state in zip(self.blocks, state.blocks):
            x, next_state = block.step(x, block_state)
            next_states.append(next_state)
        logits = self.head(self.ln_f(x)).view(idx.shape[0], self.config.n_predict,
                                               self.config.vocab_size)
        return logits[:, 0], StreamState(blocks=next_states)

    @staticmethod
    def _sample_next(logits: torch.Tensor, temperature: float, top_k: Optional[int]) -> torch.Tensor:
        if temperature <= 0:
            raise ValueError("temperature must be positive")
        logits = logits / temperature
        if top_k is not None:
            if not 1 <= top_k <= logits.shape[-1]:
                raise ValueError("top_k must be between 1 and vocab_size")
            threshold = torch.topk(logits, top_k, dim=-1).values[:, [-1]]
            logits = logits.masked_fill(logits < threshold, float('-inf'))
        return torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Correct one-byte autoregressive sampling with constant-size state."""
        was_training = self.training
        self.eval()
        try:
            logits, state = self.prefill(idx)
            for _ in range(max_new_tokens):
                idx_next = self._sample_next(logits, temperature, top_k)
                idx = torch.cat((idx, idx_next), dim=1)
                logits, state = self.step(idx_next, state)
            return idx
        finally:
            self.train(was_training)


# -----------------------------------------------------------------------------
# RetrievalBlock v2 verification (CPU, deterministic).
# -----------------------------------------------------------------------------
def check_retrieval_v2(seed: int = 1337) -> Tuple[bool, str]:
    """
    Verifies RetrievalBlock v2:
      (1) v2 with every feature OFF reproduces v1 exactly (byte-identical).
      (2) every pathway is causal: output at t is independent of inputs > t.
      (3) gradients flow to every new v2 parameter.
      (4) full Stream(StreamR) forward/backward with v2 enabled runs cleanly.
    """
    torch.manual_seed(seed)
    B, T, D_ = 2, 200, 64
    nh = 4
    torch.set_grad_enabled(True)

    msgs = []

    # (1) features-off block must equal the exact v1 formula (independent impl)
    block = RetrievalBlock(D_, n_head=nh, window=32, n_global=8,
                           per_head_bias=False, stride=0, mem_slots=0, gated=False)
    x = torch.randn(B, T, D_, requires_grad=True)
    y, _ = block(x)

    # reference: rebuild v1 math inline
    w, ng = 32, 8
    with torch.no_grad():
        q, k, v = block.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, nh, D_ // nh).transpose(1, 2)
        k_pad = F.pad(k, (0, 0, w - 1, 0))
        v_pad = F.pad(v, (0, 0, w - 1, 0))
        k_win = k_pad.unfold(1, w, 1).view(B, T, nh, D_ // nh, w).transpose(1, 2)
        v_win = v_pad.unfold(1, w, 1).view(B, T, nh, D_ // nh, w).transpose(1, 2)
        lw = torch.einsum('bhtd,bhtdw->bhtw', q, k_win) * ((D_ // nh) ** -0.5)
        dist = (w - 1) - torch.arange(w)
        lw = lw + block.rel_bias[dist].view(1, 1, 1, w)
        lg = torch.einsum('bhtd,ghd->bhtg', q, block.g_k) * ((D_ // nh) ** -0.5)
        att = torch.softmax(torch.cat([lg, lw], dim=-1), dim=-1)
        ow = torch.einsum('bhtw,bhtdw->bhtd', att[..., ng:], v_win)
        og = torch.einsum('bhtg,ghd->bhtd', att[..., :ng], block.g_v)
        y_ref = block.ln(block.proj((ow + og).transpose(1, 2).reshape(B, T, D_)) + x)
    d = (y - y_ref).abs().max().item()
    ok1 = d < 1e-6
    msgs.append(f"[1] v1-equivalence (features off): max|diff|={d:.2e} {'OK' if ok1 else 'FAIL'}")

    # (2) causality: y[t] must not depend on inputs at positions > t
    block_v2 = RetrievalBlock(D_, n_head=nh, window=32, n_global=8,
                              per_head_bias=True, stride=8, stride_slots=16,
                              mem_slots=6, mem_seg=16, gated=True)
    x2 = torch.randn(B, T, D_, requires_grad=True)
    y2, _ = block_v2(x2)
    t_cut = 50
    # gradient of output at early positions w.r.t. token at t_cut+1 (and beyond)
    loss = y2[:, :t_cut].sum()
    loss.backward(retain_graph=True)
    g_leak = x2.grad[:, t_cut + 1:].abs().max().item()
    # gradient w.r.t. an early token should be nonzero (block actually uses context)
    g_used = x2.grad[:, :t_cut].abs().max().item()
    ok2 = (g_leak < 1e-9) and (g_used > 0)
    msgs.append(f"[2] causality: leak_grad={g_leak:.2e} used_grad={g_used:.2e} "
                f"{'OK' if ok2 else 'FAIL'}")

    # (3) every new v2 param receives nonzero gradient
    block_v3 = RetrievalBlock(D_, n_head=nh, window=32, n_global=8,
                              per_head_bias=True, stride=8, stride_slots=16,
                              mem_slots=6, mem_seg=16, gated=True)
    x3 = torch.randn(B, T, D_, requires_grad=True)
    y3, _ = block_v3(x3)
    y3.pow(2).mean().backward()
    new_params = {
        'rel_bias': 'per-head bias', 'stride_bias': 'stride bias',
        'seg_q': 'segment pool query', 'mem_bias': 'memory bias',
        'path_logits': 'pathway gate',
    }
    ok3 = True
    for name, label in new_params.items():
        p = getattr(block_v3, name)
        gn = p.grad.abs().max().item() if p.grad is not None else 0.0
        ok3 &= gn > 0
        msgs.append(f"[3] grad[{name}] ({label}): {gn:.2e} {'OK' if gn > 0 else 'FAIL'}")
    if not ok3:
        msgs.append("[3] FAIL: some v2 params got zero gradient")
    else:
        msgs.append("[3] all v2 params receive gradients: OK")

    # (4) full model (StreamR) forward/backward with v2 enabled on GPU/CPU
    torch.manual_seed(seed)
    cfg = StreamConfig(n_embd=64, n_layer=2, n_predict=4, block_size=T,
                       ssm_d_state=4, n_retrieval=1, n_attn_head=nh,
                       window_size=32, n_global=8,
                       per_head_bias=True, retr_stride=8, retr_stride_slots=8,
                       retr_mem_slots=4, retr_mem_seg=16, retr_gated=True)
    stream = Stream(cfg)
    idx = torch.randint(0, 256, (B, T))
    tgt = torch.randint(0, 256, (B, T))
    logits, loss = stream(idx, targets=tgt)
    loss.backward()
    n_grad = sum(1 for p in stream.parameters() if p.grad is not None)
    n_params = sum(1 for p in stream.parameters())
    ok4 = loss is not None and torch.isfinite(loss) and n_grad == n_params
    n_tot = sum(p.numel() for p in stream.parameters())
    msgs.append(f"[4] StreamR v2 full model: loss={loss.item() if loss else float('nan'):.2f} "
                f"grads {n_grad}/{n_params} params={n_tot:,} {'OK' if ok4 else 'FAIL'}")

    # (5) chunked window pathway == full-unfold reference (multi-chunk forced)
    torch.manual_seed(seed)
    Bw, Tw, ww, nhw = 2, 200, 32, 4
    Dw = nhw * 16
    qw = torch.randn(Bw, nhw, Tw, 16)
    kw = torch.randn(Bw, Tw, Dw)
    vw = torch.randn(Bw, Tw, Dw)
    rb = torch.randn(nhw, 2 * ww - 1) * 0.1
    o_chunked = _causal_window(qw, kw, vw, ww, rb, per_head=True, chunk=48)
    k_pad = F.pad(kw, (0, 0, ww - 1, 0))
    v_pad = F.pad(vw, (0, 0, ww - 1, 0))
    k_full = k_pad.unfold(1, ww, 1).view(Bw, Tw, nhw, 16, ww).transpose(1, 2)
    v_full = v_pad.unfold(1, ww, 1).view(Bw, Tw, nhw, 16, ww).transpose(1, 2)
    lw = torch.einsum('bhtd,bhtdw->bhtw', qw, k_full) * (16 ** -0.5)
    dist_w = (ww - 1) - torch.arange(ww)
    lw = lw + rb[:, dist_w].unsqueeze(0).unsqueeze(2)
    att_f = torch.softmax(lw, dim=-1)
    o_full = torch.einsum('bhtw,bhtdw->bhtd', att_f, v_full)
    d5 = (o_chunked - o_full).abs().max().item()
    ok5 = d5 < 1e-5
    msgs.append(f"[5] chunked window vs full unfold (forced multi-chunk): "
                f"max|diff|={d5:.2e} {'OK' if ok5 else 'FAIL'}")

    ok = ok1 and ok2 and ok3 and ok4 and ok5
    summary = "\n".join(msgs) + f"\nRETRIEVAL V2 SUMMARY: {'ALL PASS' if ok else 'FAIL'}"
    return ok, summary


# -----------------------------------------------------------------------------
# DeltaMemoryBlock verification (CPU, deterministic).
# -----------------------------------------------------------------------------
def _ref_delta(block: DeltaMemoryBlock, x: torch.Tensor) -> torch.Tensor:
    """Independent re-implementation of the delta recurrence, unrolled in a
    different code structure (explicit per-step state list + gather) so the
    block's fused scan and the reference cannot share a bug."""
    B, T, D = x.shape
    nh, hd = block.n_head, block.head_dim
    with torch.no_grad():
        q, k, v = block.qkv(x).chunk(3, dim=-1)
        k_raw = k
        k = block.k_norm(k.view(B, T, nh, hd)).view(B, T, nh, hd)
        qh, vh = q.view(B, T, nh, hd), v.view(B, T, nh, hd)
        g = torch.sigmoid(block.gate_logits(x))
        lam, beta = g[..., :nh], g[..., nh:]
        S = [torch.zeros(B, nh, hd, hd)]
        preds = []
        for t in range(T):
            o = torch.einsum('bndm,bnm->bnd', S[t], qh[:, t])
            preds.append(o)
            err = vh[:, t] - torch.einsum('bndm,bnm->bnd', S[t], k[:, t])
            S.append(lam[:, t].unsqueeze(-1).unsqueeze(-1) * S[t]
                     + beta[:, t].unsqueeze(-1).unsqueeze(-1)
                     * torch.einsum('bnd,bne->bnde', k[:, t], err))
        o = torch.stack(preds, dim=1).reshape(B, T, D)
        if block.window > 0:
            # reference window pathway (mirror of block forward's layout)
            w = block.window
            k_pad = F.pad(k_raw, (0, 0, w - 1, 0))
            v_pad = F.pad(v, (0, 0, w - 1, 0))
            k_win = k_pad.unfold(1, w, 1).view(B, T, nh, hd, w).transpose(1, 2)   # (B, nh, T, hd, w)
            v_win = v_pad.unfold(1, w, 1).view(B, T, nh, hd, w).transpose(1, 2)
            q_pj = qh.transpose(1, 2)                                             # (B, nh, T, hd)
            lw = torch.matmul(q_pj.unsqueeze(-2), k_win).squeeze(-2) * (hd ** -0.5)
            dist = (w - 1) - torch.arange(w)
            lw = lw + block.rel_bias[:, dist].unsqueeze(0).unsqueeze(2)
            att = torch.softmax(lw, dim=-1)
            o_w = torch.matmul(att.unsqueeze(-2), v_win.transpose(-1, -2)).squeeze(-2)   # (B, nh, T, hd)
            wts = torch.softmax(block.path_logits.detach(), dim=0)
            o_d = o.reshape(B, T, nh, hd).transpose(1, 2)          # (B, nh, T, hd)
            o = (wts[0].view(1, nh, 1, 1) * o_d
                 + wts[1].view(1, nh, 1, 1) * o_w)
            o = o.transpose(1, 2).reshape(B, T, D)
        return block.ln(block.proj(o) + x)


def check_delta(seed: int = 1337) -> Tuple[bool, str]:
    """
    Verifies DeltaMemoryBlock:
      (1) pure-recurrence scan equals an independently unrolled reference.
      (2) strict causality: output at t is independent of inputs > t.
      (3) gradients (incl. the lambda/beta gates) match finite differences.
      (4) hybrid (window fused) path equals its independent reference.
      (5) full Stream (Stream-D) forward/backward with n_delta=2 runs cleanly.
      (6) lambda/beta gate biases stay pinned after Stream's blanket re-init.
    """
    torch.manual_seed(seed)
    B, T, D = 2, 24, 32
    nh = 4
    msgs = []

    # (1) pure recurrence vs reference
    block = DeltaMemoryBlock(D, n_head=nh, window=0)
    x = torch.randn(B, T, D, requires_grad=True)
    y, _ = block(x)
    y_ref = _ref_delta(block, x.detach())
    d1 = (y - y_ref).abs().max().item()
    ok1 = d1 < 1e-6
    msgs.append(f"[1] pure-delta scan vs reference: max|diff|={d1:.2e} {'OK' if ok1 else 'FAIL'}")

    # (2) causality: early outputs must not depend on later inputs
    x2 = torch.randn(B, T, D, requires_grad=True)
    y2, _ = block(x2)
    t_cut = 8
    y2[:, :t_cut].sum().backward(retain_graph=True)
    leak = x2.grad[:, t_cut + 1:].abs().max().item()
    used = x2.grad[:, :t_cut].abs().max().item()
    ok2 = (leak < 1e-9) and (used > 0)
    msgs.append(f"[2] causality: leak_grad={leak:.2e} used_grad={used:.2e} "
                f"{'OK' if ok2 else 'FAIL'}")

    # (3) finite-difference grads on gate logits (lambda/beta) and a linear weight
    def make():
        b = DeltaMemoryBlock(D, n_head=nh, window=0)
        b = b.double()
        return b

    block3 = make()
    x3 = torch.randn(B, T, D, dtype=torch.double, requires_grad=True)
    y3 = block3(x3)[0]
    loss3 = y3.pow(2).mean()
    loss3.backward()
    ok3 = True
    # sample params: gate_logits.bias[0] (lambda init), gate_logits.weight[0,:3], qkv.weight[0,:3]
    checks = [('gate_logits.bias', block3.gate_logits.bias.data, 0),
              ('gate_logits.weight', block3.gate_logits.weight.data, 0),
              ('qkv.weight', block3.qkv.weight.data, 0)]
    for name, pdata, idx in checks:
        src = pdata.ravel()
        target = (block3.get_parameter(name) if name != 'gate_logits.bias'
                  else block3.gate_logits.bias)
        grad = target.grad.ravel()[idx]
        eps = 1e-4 * max(abs(src[idx].item()), 1e-3)
        p0 = src[idx].item()
        src[idx] = p0 + eps
        yp = block3(x3.detach())[0].pow(2).mean().item()
        src[idx] = p0 - eps
        ym = block3(x3.detach())[0].pow(2).mean().item()
        src[idx] = p0
        fd = (yp - ym) / (2 * eps)
        rel = abs(fd - grad.item()) / (abs(fd) + abs(grad.item()) + 1e-12)
        ok3 &= rel < 5e-2
        msgs.append(f"[3] FD[{name}[{idx}]] grad={grad.item():.4e} fd={fd:.4e} rel={rel:.2e} "
                    f"{'OK' if rel < 5e-2 else 'FAIL'}")

    # (4) hybrid window path vs reference
    torch.manual_seed(seed)
    bh = DeltaMemoryBlock(D, n_head=nh, window=6)
    x4 = torch.randn(B, T, D, requires_grad=True)
    y4, _ = bh(x4)
    y4_ref = _ref_delta(bh, x4.detach())
    d4 = (y4 - y4_ref).abs().max().item()
    ok4 = d4 < 1e-6
    msgs.append(f"[4] hybrid (window fused) vs reference: max|diff|={d4:.2e} {'OK' if ok4 else 'FAIL'}")

    # (5) full Stream-D model
    torch.manual_seed(seed)
    cfg = StreamConfig(n_embd=D, n_layer=2, n_predict=4, block_size=T,
                       ssm_d_state=4, n_delta=1,
                       delta_head=nh, delta_window=0)
    stream = Stream(cfg)
    idx = torch.randint(0, 256, (B, T))
    tgt = torch.randint(0, 256, (B, T))
    logits, loss = stream(idx, targets=tgt)
    loss.backward()
    n_grad = sum(1 for p in stream.parameters() if p.grad is not None)
    n_params = sum(1 for p in stream.parameters())
    ok5 = loss is not None and torch.isfinite(loss) and n_grad == n_params
    n_tot = sum(p.numel() for p in stream.parameters())
    msgs.append(f"[5] Stream-D full model: loss={loss.item() if loss else float('nan'):.2f} "
                f"grads {n_grad}/{n_params} params={n_tot:,} {'OK' if ok5 else 'FAIL'}")

    # (6) gate-bias pinning survives blanket re-initialization
    torch.manual_seed(seed)
    cfg6 = StreamConfig(n_embd=D, n_layer=1, n_predict=2, block_size=T,
                        ssm_d_state=4, n_delta=1, delta_head=nh)
    dblock = Stream(cfg6).blocks[-1]
    lam0 = torch.sigmoid(dblock.gate_logits.bias[:nh])
    bet0 = torch.sigmoid(dblock.gate_logits.bias[nh:])
    w0 = dblock.gate_logits.weight.abs().max().item()
    exp_lam = torch.sigmoid(torch.tensor(dblock.lam_init)).item()
    exp_bet = torch.sigmoid(torch.tensor(dblock.beta_init)).item()
    ok6 = (torch.allclose(lam0, torch.full_like(lam0, exp_lam), atol=1e-5)
           and torch.allclose(bet0, torch.full_like(bet0, exp_bet), atol=1e-5)
           and w0 == 0.0)
    msgs.append(f"[6] gate-bias pinned after _init_weights: "
                f"lam={lam0[0].item():.3f} (exp {exp_lam:.3f}) beta={bet0[0].item():.3f} "
                f"(exp {exp_bet:.3f}) |W|max={w0:.1e} {'OK' if ok6 else 'FAIL'}")

    ok = ok1 and ok2 and ok3 and ok4 and ok5 and ok6
    summary = "\n".join(msgs) + f"\nDELTA SUMMARY: {'ALL PASS' if ok else 'FAIL'}"
    return ok, summary
'''
_DELTA_SRC = r'''"""
Fused gated-delta-rule scan (Triton) for DeltaMemoryBlock.

Replaces the per-timestep Python loop (~5 small kernel launches per token,
plus a fully unrolled autograd graph) with ONE sequential kernel per pass:

  forward : grid (B*nh,), recurrent matrix state W held in registers,
            writes o and the pre-update state trajectory W_{t-1}
            (needed by backward; stored fp16 under autocast).
  backward: same grid, reverse sweep, adjoint A in registers, recomputes
            err from the stored trajectory, emits gq/gk/gv/glam/gbet.

Recurrence per head (W_h in R^{hd x hd}, read-before-write = strictly causal):
    Sk_t  = W_{t-1} k_t
    o_t   = W_{t-1} q_t
    err_t = v_t - Sk_t
    W_t   = lam_t * W_{t-1} + beta_t * k_t (x) err_t

Backward math (g_t := dL/do_t, A_t := dL/dW_t accumulated from steps > t):
    c_t      = A_t^T k_t
    Ge_t     = A_t err_t
    dq_t     = W_{t-1}^T g_t
    dk_t     = beta_t (Ge_t - W_{t-1}^T c_t)
    dv_t     = beta_t c_t
    dlam_t   = <A_t, W_{t-1}>
    dbeta_t  = <c_t, err_t>
    A_{t-1}  = lam_t A_t - beta_t k_t c_t^T + g_t q_t^T

Training memory: O(T*hd^2*nh*B) for the trajectory (fp16 when autocasting),
vs. the eager path's unrolled graph of ~10 intermediates per step. Speed:
~T kernel launches -> 2, i.e. thousands of launches collapse into two.

Usage:
    from delta_scan import enable_delta_triton
    enable_delta_triton(model)          # verifies on-GPU first, safe fallback

Everything degrades gracefully: no Triton / no CUDA / hd > 64 => the eager
torch loop in model.py keeps running.
"""

import torch

try:
    import triton
    import triton.language as tl
    _HAS_TRITON = True
except Exception:
    _HAS_TRITON = False

_MAX_FUSED_HD = 64   # register-resident hd x hd fp32 accumulator


# -----------------------------------------------------------------------------
# Triton kernels (contiguous layouts enforced by the wrapper):
#   q/k/v/o : (B, T, nh, hd)   lam/beta/grads thereof : (B, T, nh)
#   straj   : (B, T, nh, hd, hd)
# -----------------------------------------------------------------------------
if _HAS_TRITON:

    @triton.jit
    def _delta_fwd_kernel(QP, KP, VP, LP, BP, OP, SP,
                          T, NH, HD,
                          BLOCK_D: tl.constexpr, EVEN_D: tl.constexpr):
        pid = tl.program_id(0)
        pb = pid // NH
        ph = pid % NH
        d = tl.arange(0, BLOCK_D)
        if EVEN_D:
            dm = d < BLOCK_D + 1  # always true; kept for shape uniformity
        else:
            dm = d < HD
        # base offsets for this (batch, head) pair
        row0 = (pb * T * NH + ph)              # first token/head offset unit
        qb = QP + row0 * HD + d                # advance t via NH*HD each step
        kb = KP + row0 * HD + d
        vb = VP + row0 * HD + d
        ob = OP + row0 * HD + d
        lb = LP + pb * T * NH + ph             # scalar gates
        bb = BP + pb * T * NH + ph
        sb = SP + row0 * HD * HD               # trajectory rows
        r2 = d[:, None] * HD + d[None, :]
        if EVEN_D:
            m2 = r2 < (HD + 1) * (HD + 1)      # always true
        else:
            m2 = (d[:, None] < HD) & (d[None, :] < HD)

        W = tl.zeros([BLOCK_D, BLOCK_D], dtype=tl.float32)
        step = NH * HD
        sstep = NH * HD * HD
        lstep = NH
        for t in range(T):
            kt = tl.load(kb + t * step, mask=dm, other=0.0).to(tl.float32)
            qt = tl.load(qb + t * step, mask=dm, other=0.0).to(tl.float32)
            vt = tl.load(vb + t * step, mask=dm, other=0.0).to(tl.float32)
            lam = tl.load(lb + t * lstep).to(tl.float32)
            bet = tl.load(bb + t * lstep).to(tl.float32)
            # save pre-update state for backward
            tl.store(sb + t * sstep + r2, W, mask=m2)
            # reads use the PREVIOUS state (strict causality)
            Sk = tl.sum(W * kt[None, :], axis=1)               # W @ k
            o = tl.sum(W * qt[None, :], axis=1)                # W @ q
            tl.store(ob + t * step, o.to(OP.dtype.element_ty), mask=dm)
            err = vt - Sk
            # gated delta write: erase along k, correct toward v
            W = lam * W + bet * kt[:, None] * err[None, :]

    @triton.jit
    def _delta_bwd_kernel(QP, KP, VP, LP, BP, GOP, SP,
                          GQP, GKP, GVP, GLP, GBP,
                          T, NH, HD,
                          BLOCK_D: tl.constexpr, EVEN_D: tl.constexpr):
        pid = tl.program_id(0)
        pb = pid // NH
        ph = pid % NH
        d = tl.arange(0, BLOCK_D)
        if EVEN_D:
            dm = d < BLOCK_D + 1
        else:
            dm = d < HD
        row0 = pb * T * NH + ph
        qb = QP + row0 * HD + d
        kb = KP + row0 * HD + d
        vb = VP + row0 * HD + d
        gb = GOP + row0 * HD + d
        lb = LP + pb * T * NH + ph
        bb = BP + pb * T * NH + ph
        sb = SP + row0 * HD * HD
        gqb = GQP + row0 * HD + d
        gkb = GKP + row0 * HD + d
        gvb = GVP + row0 * HD + d
        glb = GLP + pb * T * NH + ph
        gbb = GBP + pb * T * NH + ph
        r2 = d[:, None] * HD + d[None, :]
        if EVEN_D:
            m2 = r2 < (HD + 1) * (HD + 1)
        else:
            m2 = (d[:, None] < HD) & (d[None, :] < HD)

        A = tl.zeros([BLOCK_D, BLOCK_D], dtype=tl.float32)
        step = NH * HD
        sstep = NH * HD * HD
        lstep = NH
        for tt in range(T):
            t = T - 1 - tt
            kt = tl.load(kb + t * step, mask=dm, other=0.0).to(tl.float32)
            qt = tl.load(qb + t * step, mask=dm, other=0.0).to(tl.float32)
            vt = tl.load(vb + t * step, mask=dm, other=0.0).to(tl.float32)
            gt = tl.load(gb + t * step, mask=dm, other=0.0).to(tl.float32)
            lam = tl.load(lb + t * lstep).to(tl.float32)
            bet = tl.load(bb + t * lstep).to(tl.float32)
            S = tl.load(sb + t * sstep + r2, mask=m2, other=0.0).to(tl.float32)
            # recompute forward intermediates from the saved pre-state
            Sk = tl.sum(S * kt[None, :], axis=1)               # S @ k
            err = vt - Sk
            # adjoint pieces
            c = tl.sum(A * kt[:, None], axis=0)                # A^T @ k
            Ge = tl.sum(A * err[None, :], axis=1)              # A @ err
            St_c = tl.sum(S * c[:, None], axis=0)              # S^T @ c
            St_g = tl.sum(S * gt[:, None], axis=0)             # S^T @ g
            # input grads (all wrt the pre-update state S = W_{t-1})
            tl.store(gqb + t * step, St_g, mask=dm)
            tl.store(gkb + t * step, bet * (Ge - St_c), mask=dm)
            tl.store(gvb + t * step, bet * c, mask=dm)
            dlam = tl.sum(A * S)                               # <A, S>
            dbet = tl.sum(c * err)                             # <A^T k, err>
            tl.store(glb + t * lstep, dlam)
            tl.store(gbb + t * lstep, dbet)
            # propagate adjoint to previous state
            A = lam * A - bet * kt[:, None] * c[None, :] + gt[:, None] * qt[None, :]


def _next_pow2(n: int) -> int:
    p = 16
    while p < n:
        p *= 2
    return p


def delta_scan_fused(q, k, v, lam, beta):
    """Fused gated-delta scan.

    q, k, v: (B, T, nh, hd); lam, beta: (B, T, nh). Returns (o, None);
    the second slot mirrors _delta_scan_eager's (o, final_state) contract.
    """
    B, T, nh, hd = q.shape
    assert lam.shape == beta.shape == (B, T, nh), "gate layout mismatch"
    assert hd <= _MAX_FUSED_HD, f"hd={hd} too large for fused path"
    assert q.is_cuda, "fused delta scan requires CUDA"
    q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
    lam, beta = lam.contiguous(), beta.contiguous()
    o = torch.empty_like(q)
    traj_dt = torch.float32 if q.dtype == torch.float32 else torch.float16
    straj = torch.empty((B, T, nh, hd, hd), device=q.device, dtype=traj_dt)

    BLOCK_D = _next_pow2(hd)
    nw = 4 if hd <= 32 else 8
    grid = (B * nh,)
    _delta_fwd_kernel[grid](q, k, v, lam, beta, o, straj,
                            T, nh, hd, BLOCK_D=BLOCK_D, EVEN_D=(BLOCK_D == hd),
                            num_warps=nw)
    o = _DeltaScanFn.apply(q, k, v, lam, beta, o, straj)
    return o, None


class _DeltaScanFn(torch.autograd.Function):

    @staticmethod
    def forward(ctx, q, k, v, lam, beta, o, straj):
        ctx.save_for_backward(q, k, v, lam, beta, straj)
        return o

    @staticmethod
    def backward(ctx, go):
        q, k, v, lam, beta, straj = ctx.saved_tensors
        B, T, nh, hd = q.shape
        go = go.contiguous()
        f32 = torch.float32
        gq = torch.empty(B, T, nh, hd, device=q.device, dtype=f32)
        gk = torch.empty_like(gq)
        gv = torch.empty_like(gq)
        glam = torch.empty(B, T, nh, device=q.device, dtype=f32)
        gbet = torch.empty_like(glam)
        BLOCK_D = _next_pow2(hd)
        nw = 4 if hd <= 32 else 8
        grid = (B * nh,)
        _delta_bwd_kernel[grid](q, k, v, lam, beta, go, straj,
                                gq, gk, gv, glam, gbet,
                                T, nh, hd, BLOCK_D=BLOCK_D,
                                EVEN_D=(BLOCK_D == hd), num_warps=nw)
        need = ctx.needs_input_grad
        return (gq.to(q.dtype) if need[0] else None,
                gk.to(k.dtype) if need[1] else None,
                gv.to(v.dtype) if need[2] else None,
                glam.to(lam.dtype) if need[3] else None,
                gbet.to(beta.dtype) if need[4] else None,
                None, None)


# -----------------------------------------------------------------------------
# Verification + opt-in patching
# -----------------------------------------------------------------------------
def check_delta_triton(B=3, T=97, nh=4, hd=32, verbose=True) -> bool:
    """Compare fused kernels against a plain torch reference (CUDA only).

    Checks outputs AND gradients (q/k/v/lam/beta) incl. finite-difference
    validation of the analytic gate grads.
    """
    if not (_HAS_TRITON and torch.cuda.is_available()):
        if verbose:
            print("[check_delta_triton] skipped: Triton/CUDA unavailable")
        return False
    dev = 'cuda'
    torch.manual_seed(0)
    q = torch.randn(B, T, nh, hd, device=dev, requires_grad=True)
    k = torch.randn(B, T, nh, hd, device=dev, requires_grad=True)
    v = torch.randn(B, T, nh, hd, device=dev, requires_grad=True)
    lam = torch.sigmoid(torch.randn(B, T, nh, device=dev, requires_grad=True))
    beta = torch.sigmoid(torch.randn(B, T, nh, device=dev, requires_grad=True))

    def ref_scan(q, k, v, lam, beta):
        outs = []
        S = torch.zeros(B, nh, hd, hd, device=q.device)
        lam_ = lam.unsqueeze(-1).unsqueeze(-1)
        beta_ = beta.unsqueeze(-1).unsqueeze(-1)
        for t in range(T):
            kt, qt, vt = k[:, t], q[:, t], v[:, t]
            Sk = torch.matmul(S, kt.unsqueeze(-1)).squeeze(-1)
            outs.append(torch.matmul(S, qt.unsqueeze(-1)).squeeze(-1))
            err = vt - Sk
            S = lam_[:, t] * S + beta_[:, t] * kt.unsqueeze(-1) * err.unsqueeze(-2)
        return torch.stack(outs, dim=1)

    ok = True
    with torch.no_grad():
        o_f = delta_scan_fused(q.detach().clone().requires_grad_(False),
                               k.detach(), v.detach(),
                               lam.detach(), beta.detach())[0]
        o_r = ref_scan(q.detach(), k.detach(), v.detach(), lam.detach(), beta.detach())
    e = (o_f - o_r).abs().max().item()
    ok &= e < 1e-3
    if verbose:
        print(f"[check_delta_triton] fwd max|diff| vs reference: {e:.3e}")

    # analytic grads: fused vs autograd through the reference
    args = [q, k, v, lam, beta]
    loss_r = ref_scan(*args).pow(2).sum()
    gs = torch.autograd.grad(loss_r, args)
    o_f = delta_scan_fused(*args)[0]
    gf = torch.autograd.grad(o_f.pow(2).sum(), args)
    for name, a, b in zip(('dq', 'dk', 'dv', 'dlam', 'dbeta'), gf, gs):
        scale = max(a.abs().max().item(), b.abs().max().item(), 1e-8)
        rel = ((a - b).abs().max() / scale).item()
        good = rel < 5e-3 or a.abs().max().item() < 1e-6
        ok &= good
        if verbose:
            print(f"[check_delta_triton] grad {name}: rel_err={rel:.3e} "
                  f"{'OK' if good else 'MISMATCH'}")

    # finite-difference sanity on lambda/beta gates
    eps = 1e-3
    with torch.no_grad():
        base = ref_scan(q.detach(), k.detach(), v.detach(),
                        lam.detach(), beta.detach()).pow(2).sum().item()
    fd_ok = True
    for idx, name in ((3, 'lam'), (4, 'beta')):
        p = args[idx].detach()
        num = torch.zeros_like(p)
        it = p.flatten()
        for j in range(0, it.numel(), max(1, it.numel() // 7)):
            orig = it[j].item()
            pp = p.clone(); pp.flatten()[j] = orig + eps
            lp = ref_scan(q.detach(), k.detach(), v.detach(), pp if idx == 3 else lam.detach(),
                          beta.detach() if idx == 3 else pp).pow(2).sum().item()
            pm = p.clone(); pm.flatten()[j] = orig - eps
            lm = ref_scan(q.detach(), k.detach(), v.detach(), pm if idx == 3 else lam.detach(),
                          beta.detach() if idx == 3 else pm).pow(2).sum().item()
            num.flatten()[j] = (lp - lm) / (2 * eps)
            it[j] = orig
        ana = gf[idx].detach()
        m = num.abs() > 1e-4
        if m.any():
            rel = ((num[m] - ana[m]).abs() / (num[m].abs() + 1e-8)).median().item()
            fd_ok &= rel < 0.25
    ok &= fd_ok
    if verbose:
        print(f"[check_delta_triton] FD gate grads: {'OK' if fd_ok else 'MISMATCH'}")
        print(f"[check_delta_triton] {'ALL PASS' if ok else 'FAIL'}")
    return ok


def enable_delta_triton(model=None, verify=True, verbose=True) -> bool:
    """Opt every DeltaMemoryBlock into the fused Triton scan (CUDA only).

    Runs check_delta_triton() first unless verify=False. Returns True if any
    block was switched over; otherwise blocks keep the eager torch loop.

    Blocks are located by class name (duck-typing), NOT by importing model.py,
    so this works when the caller loaded the module under a different name
    (e.g. importlib 'md' in the Colab notebook) without creating a duplicate.
    """
    if not (_HAS_TRITON and torch.cuda.is_available()):
        if verbose:
            print("[delta_scan] Triton/CUDA unavailable -- eager loop retained")
        return False
    blocks = ([m for m in model.modules()
               if type(m).__name__ == 'DeltaMemoryBlock']
              if model is not None else [])
    hd_ok = all(b.head_dim <= _MAX_FUSED_HD for b in blocks)
    if not hd_ok:
        if verbose:
            print("[delta_scan] head_dim too large for registers -- eager kept")
        return False
    if verify and not check_delta_triton(verbose=verbose):
        return False
    for b in blocks:
        b._use_fused = True
    if verbose:
        print(f"[delta_scan] fused scan enabled on {len(blocks)} block(s)")
    return len(blocks) > 0 or model is None
'''

# --- bootstrap: write model.py fresh, then import it (drop cached module) ---
_vd = None
for _p in [os.getcwd(), '/content/RETRANS-X', '/content']:
    _v = os.path.join(_p, 'VECTOR')
    if os.path.isdir(_v): _vd = _v; break
if _vd is None:
    raise RuntimeError('no VECTOR/ dir found - run CELL 1 (clone) first')
for _name, _srctxt in (('model.py', _MODEL_SRC), ('delta_scan.py', _DELTA_SRC)):
    with open(os.path.join(_vd, _name), 'w', encoding='utf-8') as _f:
        _f.write(_srctxt)
sys.path.insert(0, _vd)
for _m in ('md', 'delta_scan'):
    sys.modules.pop(_m, None)
md = importlib.util.module_from_spec(
    (s := importlib.util.spec_from_file_location('md', os.path.join(_vd, 'model.py')))
); s.loader.exec_module(md)
import delta_scan as _ds   # same fresh copy (loaded by md via sys.path)
assert all(f in md.StreamConfig.__dataclass_fields__ for f in ('n_retrieval', 'n_delta')), \
    'embedded model.py is stale'
_ok_r, _s_r = md.check_retrieval_v2(); print('[checks] ' + _s_r.splitlines()[-1])
_ok_d, _s_d = md.check_delta();         print('[checks] ' + _s_d.splitlines()[-1])
assert _ok_r and _ok_d, 'embedded model.py failed its own checks'
print('model.py + delta_scan.py loaded (embedded, always-fresh): v2 + delta OK')

device = 'cuda'

# --- The free efficiency wins: Triton auto-scan + fp16 autocast ---
USE_FP16 = True   # fp16 autocast + GradScaler; set False if any NaN shows up
USE_TRITON = True # auto path (fused/chunked) proven on T4; set False to use JIT
try:
    import triton_scan as _ts
    _HAS_TS = getattr(_ts, 'HAS_TRITON', False)
except Exception:
    _HAS_TS = False
print(f'efficiency: USE_FP16={USE_FP16} USE_TRITON={USE_TRITON} HAS_TRITON={_HAS_TS}')
if USE_TRITON and not _HAS_TS:
    print('WARNING: Triton unavailable - falling back to JIT scan (slower)')
if USE_TRITON and _HAS_TS:
    # local triton_scan.py may be stale (a fresh clone is needed for the fp16
    # backward-kernel fix). Detect it; otherwise fp16 crashes the old kernels.
    _tsrc = open(_ts.__file__, encoding='utf-8').read()
    if 'h_prev dtype = out dtype' not in _tsrc:
        print('!!! local triton_scan.py is STALE (missing fp16 backward fix).', flush=True)
        print('    Re-run CELL 1 (git reset to origin/main) to refresh it.', flush=True)
        print('    For this run: falling back to the JIT scan so training still works.', flush=True)
        USE_TRITON = False
# --- gated delta memory: verify + enable the FUSED Triton scan (one-time) ---
_DELTA_OK = False
if USE_TRITON and torch.cuda.is_available():
    try:
        _DELTA_OK = bool(_ds.check_delta_triton(verbose=False))
        print('delta fused scan:',
              'ENABLED (kernels verified vs reference)' if _DELTA_OK
              else 'verification FAILED - eager loop retained')
    except Exception as e:
        print('delta fused scan unavailable:', type(e).__name__, e)
if not _DELTA_OK:
    print('delta fused scan: OFF - StreamD uses the eager loop')
torch.backends.cudnn.benchmark = True
# Reduce fragmentation for 10M tier on 14.5GB T4
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# --- data: reuse VECTOR/data/bytes if present, else download+prepare TinyStories ---
DATA_DIR = os.path.join(_vd, 'data', 'bytes')
TRAIN_BIN = os.path.join(DATA_DIR, 'train.bin')
VAL_BIN = os.path.join(DATA_DIR, 'val.bin')
if not (os.path.isfile(TRAIN_BIN) and os.path.isfile(VAL_BIN)):
    print('byte dataset missing - downloading TinyStories + preparing (one-time)...')
    import requests
    os.makedirs(DATA_DIR, exist_ok=True)
    src = os.path.join(DATA_DIR, 'TinyStories-train.txt')
    if not os.path.isfile(src):
        url = 'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt'
        print('Downloading ~940MB TinyStories-train.txt...')
        r = requests.get(url, stream=True)
        with open(src, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1 << 16):
                f.write(chunk)
    with open(src, 'rb') as f:
        raw = f.read()
    split = int(len(raw) * 0.99)
    np.frombuffer(raw[:split], dtype=np.uint8).copy().tofile(TRAIN_BIN)
    np.frombuffer(raw[split:], dtype=np.uint8).copy().tofile(VAL_BIN)
    print(f'train {split:,}B / val {len(raw)-split:,}B')
train_data = np.memmap(TRAIN_BIN, dtype=np.uint8, mode='r')
val_data = np.memmap(VAL_BIN, dtype=np.uint8, mode='r')

# --- GPT baseline: byte-level transformer with RoPE (position-free, no wpe growth) ---
def _rope(positions, dim, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, dim, 2, device=positions.device).float() / dim))
    ang = positions.float().unsqueeze(-1) * inv.unsqueeze(0)   # (T, dim/2)
    return torch.cos(ang), torch.sin(ang)

def _rot(x):
    a, b = x.chunk(2, dim=-1)
    return torch.cat((-b, a), dim=-1)

class GPTAttn(nn.Module):
    def __init__(self, n_embd, n_head, bias):
        super().__init__()
        self.n_head = n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=bias)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=bias)
    def forward(self, x, pos):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=-1)
        hd = C // self.n_head
        cos, sin = _rope(pos, hd)   # angles over the head dim (hd), not full n_embd
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        q = q * cos.unsqueeze(0).unsqueeze(0) + _rot(q) * sin.unsqueeze(0).unsqueeze(0)
        k = k * cos.unsqueeze(0).unsqueeze(0) + _rot(k) * sin.unsqueeze(0).unsqueeze(0)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class GPTMLP(nn.Module):
    def __init__(self, n_embd, bias):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd, bias=bias)
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=bias)
    def forward(self, x):
        return self.c_proj(F.gelu(self.c_fc(x)))

class GPTBlock(nn.Module):
    def __init__(self, n_embd, n_head, bias):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd, bias=bias)
        self.attn = GPTAttn(n_embd, n_head, bias)
        self.ln2 = nn.LayerNorm(n_embd, bias=bias)
        self.mlp = GPTMLP(n_embd, bias)

class GPTByte(nn.Module):
    def __init__(self, n_embd=128, n_layer=3, n_head=4, n_predict=4, vocab_size=256, bias=False):
        super().__init__()
        self.n_predict = n_predict
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([GPTBlock(n_embd, n_head, bias) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd, bias=bias)
        self.head = nn.Linear(n_embd, n_predict * vocab_size, bias=False)
        self.apply(self._init)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))
    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.wte(idx)
        pos = torch.arange(T, device=idx.device)
        for blk in self.blocks:
            x = x + blk.attn(blk.ln1(x), pos)
            x = x + blk.mlp(blk.ln2(x))
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            np_, vs = self.n_predict, logits.shape[-1] // self.n_predict
            lg = logits.view(B, T, np_, vs)
            loss = sum(F.cross_entropy(lg[:, :T-k, k].reshape(-1, vs),
                                       targets[:, k:].reshape(-1), ignore_index=-1)
                       for k in range(np_)) / np_
        return logits, loss

# --- shared data + eval ---
def make_batches(data, T, B, n, gen):
    L = len(data)
    out = []
    for _ in range(n):
        ix = torch.randint(L - T - 1, (B,), generator=gen)
        x = torch.stack([torch.from_numpy(data[i:i+T].astype(np.int64)) for i in ix]).to(device)
        y = torch.stack([torch.from_numpy(data[i+1:i+1+T].astype(np.int64)) for i in ix]).to(device)
        out.append((x, y))
    return out

def eval_loss(model, batches):
    model.eval()
    tot, n = 0.0, 0
    with torch.no_grad():
        if USE_FP16:
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                for x, y in batches:
                    _, loss = model(x, targets=y)
                    tot += loss.item(); n += 1
        else:
            for x, y in batches:
                _, loss = model(x, targets=y)
                tot += loss.item(); n += 1
    model.train()
    return tot / n

def med(ts, drop=5):
    t = sorted(ts[drop:])
    n = len(t)
    return t[n // 2] if n % 2 else (t[n // 2 - 1] + t[n // 2]) / 2

def train_model(name, model, train_batches, val_batches, steps, warmup, eval_at,
                peak_lr=6e-4, min_lr=6e-5):
    model.train()
    decay = [p for p in model.parameters() if p.dim() >= 2]
    nodecay = [p for p in model.parameters() if p.dim() < 2]
    opt = torch.optim.AdamW(
        [{'params': decay, 'weight_decay': 0.1},
         {'params': nodecay, 'weight_decay': 0.0}],
        lr=peak_lr, betas=(0.9, 0.95), fused=True)
    log_interval = max(10, steps // 12)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_FP16)
    times, evals = [], []
    t_start = time.perf_counter()
    print(f'  training {name}: {steps} steps (progress every {log_interval})...', flush=True)
    for it in range(steps):
        if it < warmup:
            lr = peak_lr * (it + 1) / warmup
        else:
            r = (it - warmup) / max(1, steps - warmup)
            lr = min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * r))
        for pg in opt.param_groups: pg['lr'] = lr
        x, y = train_batches[it]
        torch.cuda.synchronize(); t0 = time.perf_counter()
        if USE_FP16:
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                _, loss = model(x, targets=y)
            scaler.scale(loss).backward()
        else:
            _, loss = model(x, targets=y)
            loss.backward()
        if USE_FP16:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        opt.zero_grad(set_to_none=True)
        torch.cuda.synchronize(); times.append(time.perf_counter() - t0)
        if (it + 1) % log_interval == 0:
            el = time.perf_counter() - t_start
            print(f'    {name}: step {it+1}/{steps} loss {loss.item():.4f} lr {lr:.1e} ({el:.0f}s elapsed)', flush=True)
        if it in eval_at:
            evals.append((it + 1, eval_loss(model, val_batches)))
    return evals, times

def run_leg(T, steps, warmup, B, tag, skip=()):
    print(f'\n{"="*70}', flush=True)
    print(f'LEG {tag}: T={T}, B={B}, steps={steps}  (identical data for all models)', flush=True)
    print(f'{"="*70}', flush=True)
    tr = make_batches(train_data, T, B, steps, torch.Generator().manual_seed(2026))
    va = make_batches(val_data, T, B, 60, torch.Generator().manual_seed(1))
    eval_at = set([steps // 4, steps // 2, steps - 1])
    models = {}
    builders = {
        'Stream-10M': lambda: md.Stream(md.StreamConfig(
            n_embd=256, n_layer=16, n_retrieval=0, ssm_d_state=16, n_predict=4,
            block_size=T, dropout=0.0, bias=False, activation_checkpointing=True)).to(device),
        'StreamR-10M': lambda: md.Stream(md.StreamConfig(
            n_embd=256, n_layer=16, n_retrieval=2, ssm_d_state=16, n_predict=4,
            block_size=T, dropout=0.0, bias=False, n_attn_head=4,
            window_size=128, n_global=16,
            per_head_bias=True, retr_stride=8, retr_stride_slots=32,
            retr_mem_slots=16, retr_mem_seg=64, retr_gated=True)).to(device),
        'StreamD-10M': lambda: md.Stream(md.StreamConfig(
            n_embd=256, n_layer=16, n_delta=2, ssm_d_state=16, n_predict=4,
            block_size=T, dropout=0.0, bias=False, delta_head=4,
            delta_window=0, delta_lam_init=2.2, delta_beta_init=0.0, activation_checkpointing=True)).to(device),
        'GPT-8L': lambda: GPTByte(n_embd=256, n_layer=8, n_head=4, n_predict=4).to(device),
    }
    for name, fn in builders.items():
        if name in skip:
            print(f'{name}: skipped at this T (see note)', flush=True)
            continue
        torch.manual_seed(0)
        m = fn()
        if USE_TRITON and _HAS_TS and isinstance(m, md.Stream):
            _ts.enable_triton(m, auto=True)   # shape-conditional fused/chunked scan
        if _DELTA_OK and isinstance(m, md.Stream):
            _ds.enable_delta_triton(m, verify=False)  # verified once above
        n = sum(p.numel() for p in m.parameters())
        print(f'{name}: {n/1e6:.3f}M params', flush=True)
        evals, times = train_model(name, m, tr, va, steps, warmup, eval_at=eval_at)
        models[name] = dict(evals=evals, times=times, params=n)
        print(f'  -> done {name}: per-step median {med(times)*1000:.1f}ms | '
              f'final val {evals[-1][1]:.4f} @ step {evals[-1][0]}', flush=True)
    print()
    for name, d in models.items():
        print(f'  {name:12s} val: ' + ' | '.join(f'step {it} = {vl:.4f}' for it, vl in d['evals']))
    for name, d in models.items():
        print(f'  {name:12s} step median: {med(d["times"])*1000:.1f} ms')
    return models

import gc
torch.cuda.empty_cache(); gc.collect()
# --- run both legs ---
# GPT is skipped at T=16384 on purpose: its O(n^2) attention does not fit a T4
# at that length (scores alone ~2GB fp16), which is exactly the gap Stream
# exists to close. Long-context legs = Stream vs StreamR vs Stream-D.
leg1 = run_leg(4096, 1200, 100, B=4, tag='A quality')
leg2 = run_leg(16384, 400, 50, B=2, tag='B long-context', skip=('GPT-8L',))

# --- summary + verdict ---
def row(leg, name):
    d = leg.get(name)
    return (d['evals'][-1][1], med(d['times'])) if d else (float('nan'), float('nan'))

print(f'\n{"="*70}', flush=True)
print('SUMMARY (final val loss / per-step median ms)', flush=True)
print(f'{"="*70}', flush=True)
for tag, leg in [('T=4096', leg1), ('T=16384', leg2)]:
    sl, st = row(leg, 'Stream-10M')
    rl, rt = row(leg, 'StreamR-10M')
    dl, dt = row(leg, 'StreamD-10M')
    gl, gt = row(leg, 'GPT-8L')
    print(f'  {tag}:')
    print(f'    val loss : Stream {sl:.4f} | StreamR {rl:.4f} | StreamD {dl:.4f} | GPT {gl:.4f}')
    print(f'    step ms  : Stream {st*1000:.1f} | StreamR {rt*1000:.1f} | StreamD {dt*1000:.1f} | GPT {gt*1000:.1f}')
    if tag == 'T=4096':
        print(f'    bounded retrieval helps (StreamR < Stream): {rl < sl} | '
              f'    delta helps (StreamD < Stream): {dl < sl} | '
              f'    content-addressed beats bounded (StreamD < StreamR): {dl < rl} | '
              f'    gap-to-GPT closed: {min(sl,rl,dl)-gl:.4f}')
    else:
        print(f'    bounded retrieval helps (StreamR < Stream): {rl < sl} | '
              f'    delta helps (StreamD < Stream): {dl < sl} | '
              f'    content-addressed beats bounded (StreamD < StreamR): {dl < rl} | '
              f'    GPT did NOT fit T=16384 on T4 (O(n^2) attention) - the gap this model is built to close.')
